In [ ]:
# ============================================================
# LIME EXPLANATION (all 4 classes, per sample image)
# ============================================================
!pip install lime --quiet

import numpy as np
import matplotlib.pyplot as plt
from lime import lime_image
from skimage.segmentation import mark_boundaries

# ---- predict function (black-box, works regardless of nested-model graph issues) ----
def predict_fn(images):
    images = np.array(images, dtype=np.float32)
    return model.predict(images, verbose=0)

# ---- pick 1 (or more) sample per class so all 4 classes are covered ----
SAMPLES_PER_CLASS = 1  # dorkar hole barao
val_labels_arr = np.array(val_labels)
selected_indices = []
for class_idx in range(len(class_names)):
    class_indices = np.where(val_labels_arr == class_idx)[0]
    if len(class_indices) == 0:
        print(f"Warning: no validation samples for class '{class_names[class_idx]}'")
        continue
    selected_indices.extend(class_indices[:SAMPLES_PER_CLASS].tolist())

lime_explainer = lime_image.LimeImageExplainer()

LIME_NUM_SAMPLES = 1000   # perturbation koyta banabe - komale speed barbe, quality kombe
LIME_NUM_FEATURES = 8     # koyta top superpixel highlight korbe protita class e

n_rows = len(selected_indices)
n_cols = len(class_names) + 1  # 1 for original + 1 per class
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)

lime_confidence_records = []

for row, idx in enumerate(selected_indices):
    path, true_label = val_paths[idx], val_labels[idx]
    img, _ = load_image(path, true_label)
    img_np = img.numpy()

    preds = predict_fn(img_np[np.newaxis, ...])[0]
    pred_idx = int(np.argmax(preds))

    explanation = lime_explainer.explain_instance(
        img_np.astype(np.double),
        predict_fn,
        top_labels=len(class_names),
        hide_color=0,
        num_samples=LIME_NUM_SAMPLES
    )

    axes[row, 0].imshow(img_np.astype(np.uint8))
    axes[row, 0].set_title(f"Original\nTrue: {class_names[true_label]}\nPred: {class_names[pred_idx]} ({preds[pred_idx]*100:.2f}%)")
    axes[row, 0].axis("off")

    for c in range(len(class_names)):
        temp, mask = explanation.get_image_and_mask(
            c, positive_only=True, num_features=LIME_NUM_FEATURES, hide_rest=False
        )
        vis = mark_boundaries(np.clip(temp, 0, 255) / 255.0, mask)
        axes[row, c + 1].imshow(vis)
        axes[row, c + 1].set_title(f"{class_names[c]}\nconfidence: {preds[c]*100:.2f}%")
        axes[row, c + 1].axis("off")

        lime_confidence_records.append({
            "image_index": idx,
            "true_label": class_names[true_label],
            "explained_class": class_names[c],
            "confidence": float(preds[c])
        })

plt.tight_layout()
plt.savefig("/kaggle/working/lime_all_classes.png", dpi=600)
plt.show()

pd.DataFrame(lime_confidence_records).to_csv("/kaggle/working/lime_confidence.csv", index=False)
print("Saved: lime_all_classes.png, lime_confidence.csv")


In [ ]:
# ============================================================
# LIME EXPLANATION (all 4 classes, per sample image) -- FIXED
#
# Fixes vs. original Cell 4:
#   1. Caption now clearly says "Model confidence" (classifier
#      output) instead of implying it's LIME's own output.
#   2. Added LIME's own metric: local surrogate model FIDELITY
#      (R^2 score of the linear model LIME fits), taken from
#      explanation.score[label] -- this is what actually belongs
#      to LIME, not the raw softmax value.
#   3. CSV now has both columns side by side so you can show the
#      difference clearly in the paper if needed.
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lime import lime_image
from skimage.segmentation import mark_boundaries

def predict_fn(images):
    images = np.array(images, dtype=np.float32)
    return model.predict(images, verbose=0)

SAMPLES_PER_CLASS = 1  # dorkar hole barao
val_labels_arr = np.array(val_labels)
selected_indices = []
for class_idx in range(len(class_names)):
    class_indices = np.where(val_labels_arr == class_idx)[0]
    if len(class_indices) == 0:
        print(f"Warning: no validation samples for class '{class_names[class_idx]}'")
        continue
    selected_indices.extend(class_indices[:SAMPLES_PER_CLASS].tolist())

lime_explainer = lime_image.LimeImageExplainer()

LIME_NUM_SAMPLES = 1000
LIME_NUM_FEATURES = 8

n_rows = len(selected_indices)
n_cols = len(class_names) + 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)

lime_records = []

for row, idx in enumerate(selected_indices):
    path, true_label = val_paths[idx], val_labels[idx]
    img, _ = load_image(path, true_label)
    img_np = img.numpy()

    preds = predict_fn(img_np[np.newaxis, ...])[0]
    pred_idx = int(np.argmax(preds))

    explanation = lime_explainer.explain_instance(
        img_np.astype(np.double),
        predict_fn,
        top_labels=len(class_names),
        hide_color=0,
        num_samples=LIME_NUM_SAMPLES
    )

    axes[row, 0].imshow(img_np.astype(np.uint8))
    axes[row, 0].set_title(
        f"Original\nTrue: {class_names[true_label]}\n"
        f"Pred: {class_names[pred_idx]} ({preds[pred_idx]*100:.2f}%)"
    )
    axes[row, 0].axis("off")

    for c in range(len(class_names)):
        temp, mask = explanation.get_image_and_mask(
            c, positive_only=True, num_features=LIME_NUM_FEATURES, hide_rest=False
        )
        vis = mark_boundaries(np.clip(temp, 0, 255) / 255.0, mask)

        # LIME's own metric: R^2 fidelity of the local linear surrogate model
        # (how well the simple local model approximates the black-box model
        #  in the neighborhood of this instance, for this class).
        # Different lime versions store `explanation.score` differently:
        #   - dict {label: score}  -> newer versions, per-class
        #   - single float          -> older versions, only for the last
        #                              explained label (usually top_labels[0])
        # Handle both safely.
        score_attr = getattr(explanation, "score", None)
        if isinstance(score_attr, dict):
            fidelity = float(score_attr.get(c, np.nan))
        elif isinstance(score_attr, (int, float)):
            # only meaningful for the class LIME actually fit last -- usually
            # the top predicted class. For other classes we don't have a
            # per-class score in this lime version, so mark as NaN.
            fidelity = float(score_attr) if c == pred_idx else np.nan
        else:
            fidelity = np.nan

        fidelity_str = f"{fidelity:.3f}" if not np.isnan(fidelity) else "N/A"
        axes[row, c + 1].imshow(vis)
        axes[row, c + 1].set_title(
            f"{class_names[c]}\n"
            f"Model conf: {preds[c]*100:.2f}%\n"
            f"LIME fidelity (R2): {fidelity_str}"
        )
        axes[row, c + 1].axis("off")

        lime_records.append({
            "image_index": idx,
            "true_label": class_names[true_label],
            "pred_label": class_names[pred_idx],
            "explained_class": class_names[c],
            "model_confidence": float(preds[c]),   # classifier's own softmax output
            "lime_fidelity_r2": fidelity,           # LIME's own local-model quality metric
        })

plt.tight_layout()
plt.savefig("/kaggle/working/lime_all_classes.png", dpi=600)
plt.show()

pd.DataFrame(lime_records).to_csv("/kaggle/working/lime_confidence.csv", index=False)
print("Saved: lime_all_classes.png, lime_confidence.csv")
print("\nNote: 'model_confidence' = classifier softmax output (NOT LIME's output).")
print("      'lime_fidelity_r2'  = LIME's own local surrogate model quality (R^2).")
print("      NaN mane ei lime version-e oi class-er jonno alada score store hoyni")
print("      (only the predicted class's fidelity is available in this version).")

In [ ]:
# ============================================================
# LIME - INSERTION / DELETION AUC (patch-based causal metric)
# Session-resumable checkpoint. LIME explain_instance already slow,
# tai eikhane by-default same small per-class sample set use kora hocche
# (selected_indices theke). Barate chaile N_EVAL_SAMPLES_LIME barao,
# kintu shomoy onek barbe.
# ============================================================
import os
import cv2
import numpy as np
import pandas as pd
from lime import lime_image

CHECKPOINT_PATH_LIME = "/kaggle/working/lime_causal_metrics_progress.csv"
SUMMARY_PATH_LIME = "/kaggle/working/lime_causal_metrics_summary.csv"

PATCH_SIZE = 16
N_STEPS = 20
LIME_NUM_SAMPLES = 1000
LIME_NUM_FEATURES = 8

# ---- utilities (redefined here so this cell runs standalone too) ----
def get_patch_order(heatmap_224, patch_size=16):
    h, w = heatmap_224.shape
    n_h, n_w = h // patch_size, w // patch_size
    patch_scores = np.zeros((n_h, n_w))
    for i in range(n_h):
        for j in range(n_w):
            patch_scores[i, j] = heatmap_224[
                i*patch_size:(i+1)*patch_size, j*patch_size:(j+1)*patch_size
            ].mean()
    flat_order = np.argsort(-patch_scores.ravel())
    order = np.stack(np.unravel_index(flat_order, patch_scores.shape), axis=1)
    return order

def causal_metric(img_224, heatmap_224, pred_index, mode="insertion",
                   patch_size=16, n_steps=20):
    order = get_patch_order(heatmap_224, patch_size)
    n_patches = len(order)
    step = max(1, n_patches // n_steps)

    blurred = cv2.GaussianBlur(img_224, (31, 31), 0)
    canvas = blurred.copy() if mode == "insertion" else img_224.copy()

    scores, fractions = [], []
    for idx in range(0, n_patches + step, step):
        idx = min(idx, n_patches)
        for (pi, pj) in order[:idx]:
            r0, r1 = pi*patch_size, (pi+1)*patch_size
            c0, c1 = pj*patch_size, (pj+1)*patch_size
            if mode == "insertion":
                canvas[r0:r1, c0:c1] = img_224[r0:r1, c0:c1]
            else:
                canvas[r0:r1, c0:c1] = blurred[r0:r1, c0:c1]
        pred = model.predict(canvas[np.newaxis, ...], verbose=0)[0]
        scores.append(pred[pred_index])
        fractions.append(idx / n_patches)
        if idx == n_patches:
            break
    return float(np.trapz(scores, fractions))

def lime_to_heatmap(explanation, label, img_shape):
    segments = explanation.segments
    heatmap = np.zeros(img_shape[:2], dtype=np.float32)
    for seg_id, weight in explanation.local_exp[label]:
        heatmap[segments == seg_id] = weight
    heatmap = np.maximum(heatmap, 0)   # positive (supporting) evidence only
    if heatmap.max() > 0:
        heatmap = heatmap / heatmap.max()
    return heatmap

def predict_fn(images):
    images = np.array(images, dtype=np.float32)
    return model.predict(images, verbose=0)

# ---- eval set ----
N_EVAL_SAMPLES_LIME = len(selected_indices)   # ei list ta age visualization cell e create hoyeche
eval_indices = selected_indices[:N_EVAL_SAMPLES_LIME]

if os.path.exists(CHECKPOINT_PATH_LIME):
    results_df = pd.read_csv(CHECKPOINT_PATH_LIME)
    done_indices = set(results_df["image_index"].tolist())
    print(f"Resuming... {len(done_indices)}/{len(eval_indices)} done")
else:
    results_df = pd.DataFrame(columns=[
        "image_index", "true_label", "pred_label", "confidence",
        "LIME_Insertion_AUC", "LIME_Deletion_AUC"
    ])
    done_indices = set()

lime_explainer_auc = lime_image.LimeImageExplainer()

for idx in eval_indices:
    if idx in done_indices:
        continue

    path, true_label = val_paths[idx], val_labels[idx]
    img, _ = load_image(path, true_label)
    img_np = img.numpy()

    preds = predict_fn(img_np[np.newaxis, ...])[0]
    pred_idx = int(np.argmax(preds))
    confidence = float(preds[pred_idx])

    explanation = lime_explainer_auc.explain_instance(
        img_np.astype(np.double),
        predict_fn,
        top_labels=len(class_names),
        hide_color=0,
        num_samples=LIME_NUM_SAMPLES
    )

    heatmap = lime_to_heatmap(explanation, pred_idx, img_np.shape)

    ins_auc = causal_metric(img_np, heatmap, pred_idx, "insertion", PATCH_SIZE, N_STEPS)
    del_auc = causal_metric(img_np, heatmap, pred_idx, "deletion", PATCH_SIZE, N_STEPS)

    results_df.loc[len(results_df)] = [
        idx, class_names[true_label], class_names[pred_idx], confidence, ins_auc, del_auc
    ]
    results_df.to_csv(CHECKPOINT_PATH_LIME, index=False)
    print(f"[{idx}] true={class_names[true_label]} pred={class_names[pred_idx]} "
          f"conf={confidence*100:.2f}%  Ins={ins_auc:.4f}  Del={del_auc:.4f}")

summary = pd.DataFrame({
    "Mean_Confidence":    [results_df["confidence"].mean()],
    "LIME_Insertion_AUC": [results_df["LIME_Insertion_AUC"].mean()],
    "LIME_Deletion_AUC":  [results_df["LIME_Deletion_AUC"].mean()],
})
summary.to_csv(SUMMARY_PATH_LIME, index=False)
print("\n=== LIME Causal Faithfulness Summary ===")
print(summary.T)
